# 5.0 - Adult Counterfactual Sampling using CertCFAtlas (Modular API)

This notebook tests the novel method adapter `CertCF` inside the modular `counterfactuals` framework.

Goals:
- load the trained Adult classifier and switch to embedding-space classifier (`model.net`)
- downsample embedded training points with class-wise k-medoids
- build `CertCFAtlas` and wrap it with `CertCF`
- generate and evaluate one counterfactual with shared modular metrics

In [1]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import torch
from torch.utils.data import TensorDataset

from training.lit_classifier import LitClassifier
from training.datamodules.adult import CARDINALITIES, INPUT_TYPES
from models.classifiers import TabularClassifier
from certcf import CertCFAtlas, NearestOppositeClassClearanceStrategy

from counterfactuals.core.base_classes import CounterfactualExample
from counterfactuals.datasets.loaders import AdultDataset
from counterfactuals.methods.certcf import CertCF
from counterfactuals.metrics.plausibility import KNNPlausibility
from counterfactuals.metrics.proximity import L1Proximity, L2Proximity
from counterfactuals.metrics.sparsity import SparsityMetric
from counterfactuals.metrics.validity import ValidityMetric
from counterfactuals.models.torch_model import TorchModelWrapper

DEVICE = 'cpu'
CKPT = '../checkpoints/adult_classifier/last-v3.ckpt'

In [2]:
# Load Adult classifier and expose both full model and embedding classifier.
backbone = TabularClassifier(
    input_types=INPUT_TYPES,
    cardinalities=CARDINALITIES,
    embedding_dim=1,
    hidden_dims=[64, 32],
    num_classes=2,
)

lit = LitClassifier.load_from_checkpoint(CKPT, model=backbone, map_location=DEVICE)
model = lit.model.eval().to(DEVICE)
model_net_api = TorchModelWrapper(model=model.net, device=DEVICE)

print(f'embed_dim : {model.embed_dim}')
print(f'net       : {model.net}')

embed_dim : 14
net       : Sequential(
  (0): Linear(in_features=14, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=2, bias=True)
)


In [3]:
# Load Adult data in raw tabular space.
adult = AdultDataset(data_dir='../data/', seed=42)
adult.load()

x_train_np, y_train_np = adult.get_train()
x_test_np, y_test_np = adult.get_test()

print(f'Raw train: {x_train_np.shape}, Raw test: {x_test_np.shape}')

Raw train: (39074, 14), Raw test: (4884, 14)


In [4]:
# Embed train/test points and downsample train embeddings with class-wise k-medoids.
with torch.no_grad():
    z_train_np = model.embed(torch.from_numpy(x_train_np).to(DEVICE)).cpu().numpy().astype(np.float32)
    z_test_np = model.embed(torch.from_numpy(x_test_np).to(DEVICE)).cpu().numpy().astype(np.float32)

try:
    from sklearn_extra.cluster import KMedoids
    kmedoids_backend = 'sklearn_extra'
except ImportError:
    from sklearn.cluster import KMeans
    from sklearn.metrics import pairwise_distances
    KMedoids = None
    kmedoids_backend = 'kmeans+nn'

print(f'K-medoids backend: {kmedoids_backend}')

SEED = 42
K_PER_CLASS = 500

def fit_kmedoids(X: np.ndarray, k: int, seed: int = SEED):
    k = min(k, len(X))
    if KMedoids is not None:
        km = KMedoids(n_clusters=k, metric='euclidean', method='alternate', random_state=seed)
        km.fit(X)
        return km.medoid_indices_
    km = KMeans(n_clusters=k, random_state=seed, n_init='auto')
    km.fit(X)
    return pairwise_distances(km.cluster_centers_, X).argmin(axis=1)

medoid_z_parts, medoid_y_parts = [], []
for cls in np.unique(y_train_np):
    cls_idx = np.where(y_train_np == cls)[0]
    z_cls = z_train_np[cls_idx]
    med_local_idx = fit_kmedoids(z_cls, K_PER_CLASS)
    medoid_z_parts.append(torch.from_numpy(z_cls[med_local_idx]).float())
    medoid_y_parts.append(torch.full((len(med_local_idx),), int(cls), dtype=torch.long))
    print(f'Class {int(cls)}: {len(cls_idx)} points -> {len(med_local_idx)} medoids')

medoid_z_all = torch.cat(medoid_z_parts, dim=0)
medoid_y_all = torch.cat(medoid_y_parts, dim=0)
medoid_ds = TensorDataset(medoid_z_all, medoid_y_all)

print(f'Embedded train (full): {z_train_np.shape}')
print(f'Embedded train (medoids): {tuple(medoid_z_all.shape)}')
print(f'Embedded test: {z_test_np.shape}')

K-medoids backend: sklearn_extra


/home/gabrielepintus/.conda/envs/py13/lib/python3.13/site-packages/sklearn_extra/cluster/_commonnn.py:18: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  if LooseVersion(sklearn.__version__) < LooseVersion("0.23.0"):


Class 0: 29730 points -> 500 medoids
Class 1: 9344 points -> 500 medoids
Embedded train (full): (39074, 14)
Embedded train (medoids): (1000, 14)
Embedded test: (4884, 14)


In [ ]:
# Build CertCFAtlas on the embedding classifier input space.
# Use the simple nearest-opposite-class clearance heuristic for per-point epsilon.
eps_strategy = NearestOppositeClassClearanceStrategy(alpha=0.25)
atlas = CertCFAtlas(model.net, medoid_ds, device=DEVICE)
atlas.build(
    eps_strategy=eps_strategy,
    norm=1,
    batch_size=256,
)
print(atlas.summary())

Building certified atlas (eps in [0.015, 8.927] (NearestOppositeClassClearanceStrategy), L1 norm)...
  Computing LiRPA bounds...


Computing bounds:   0%|          | 0/2 [00:00<?, ?it/s]/home/gabrielepintus/.conda/envs/py13/lib/python3.13/site-packages/auto_LiRPA/perturbations.py:199: RuntimeWarning: divide by zero encountered in scalar divide
  self.dual_norm = 1 if (norm == np.inf) else (np.float64(1.0) / (1 - 1.0 / self.norm))
/home/gabrielepintus/.conda/envs/py13/lib/python3.13/site-packages/auto_LiRPA/operators/linear.py:650: RuntimeWarning: divide by zero encountered in scalar divide
  dual_norm = np.float64(1.0) / (1 - 1.0 / norm)
Computing bounds: 100%|██████████| 2/2 [00:09<00:00,  4.75s/it]

  Building BVH spatial indices...
    Class 0: 500 polytopes, tree depth 10
    Class 1: 500 polytopes, tree depth 10
Done! Total: 1000 polytopes across 2 classes
CertCFAtlas Summary
Classes: 2
Perturbation: eps in [0.015, 8.927] (NearestOppositeClassClearanceStrategy), L1 norm
Device: cpu

  Class 0:  500 polytopes (BVH depth 10)
  Class 1:  500 polytopes (BVH depth 10)

Total polytopes: 1000


## Fit and Generate with CertCF

In [16]:
atlas_method = CertCF(atlas=atlas, random_seed=42)
atlas_method.fit(x_train=z_train_np, y_train=y_train_np, model=model_net_api)
print('CertCF fitted.')

CertCF fitted.


In [17]:
# Pick one factual example predicted as class 0 in embedding space and target class 1.
pred_test = model_net_api.predict(z_test_np)
idx = int(np.where(pred_test == 0)[0][0])
z_fact = z_test_np[idx]

example = CounterfactualExample(x=z_fact, target_class=1)
atlas_result = atlas_method.generate(example=example, model=model_net_api)

print(f'Factual index: {idx}')
print(f'Factual predicted class: {int(model_net_api.predict(z_fact)[0])}')
print(f'Counterfactual predicted class: {int(model_net_api.predict(atlas_result.x_cf)[0])}')
print(f'Success: {atlas_result.success}')
print(f'L2 distance: {atlas_result.distance:.4f}')

Factual index: 0
Factual predicted class: 0
Counterfactual predicted class: 1
Success: True
L2 distance: 3.7880


## Quantitative Evaluation with Modular Metrics

In [18]:
metrics = [
    L2Proximity(),
    L1Proximity(),
    SparsityMetric(atol=1e-5),
    ValidityMetric(model=model_net_api),
    KNNPlausibility(x_train=z_train_np, n_neighbors=5),
]

context = {'target_class': 1}
results = {metric.name: metric.evaluate(z_fact, atlas_result.x_cf, context=context) for metric in metrics}

print('CertCFAtlas metrics (modular evaluator, embedding space):')
for key, value in results.items():
    print(f'  {key:15s}: {value:.6f}')

CertCFAtlas metrics (modular evaluator, embedding space):
  proximity_l2   : 3.788004
  proximity_l1   : 8.955252
  sparsity       : 0.571429
  validity       : 1.000000
  plausibility   : 0.669353


In [19]:
print('CounterfactualResult fields:')
print(f'  success  : {atlas_result.success}')
print(f'  distance : {atlas_result.distance:.6f}')
print(f'  x_cf shape: {atlas_result.x_cf.shape}')
print('  metadata :')
for key, value in atlas_result.metadata.items():
    print(f'    {key}: {value}')

CounterfactualResult fields:
  success  : True
  distance : 3.788004
  x_cf shape: (14,)
  metadata :
